In [1]:
import torch
from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

PyTorch: 2.14.0+cu126
CUDA: True
GPU: NVIDIA GeForce GTX 1050 Ti
VRAM: 4.0 GB


In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

SEGMENTATION_CONFIG = (
    PROJECT_DIR
    / "configs"
    / "aerovision_segmentation.yaml"
)

print("Projeto:", PROJECT_DIR)
print("Configuração:", SEGMENTATION_CONFIG)
print("Existe:", SEGMENTATION_CONFIG.exists())

Projeto: c:\Users\Andrezinho\OneDrive\Documentos\GitHub\AeroVision
Configuração: c:\Users\Andrezinho\OneDrive\Documentos\GitHub\AeroVision\configs\aerovision_segmentation.yaml
Existe: True


In [3]:
model = YOLO("yolo11n-seg.pt")

print("Modelo YOLO11n-seg carregado com sucesso.")

Modelo YOLO11n-seg carregado com sucesso.


In [13]:
for split in ["train", "val", "test"]:
    images_dir = SEGMENTATION_DIR / "images" / split
    labels_dir = SEGMENTATION_DIR / "labels" / split

    image_count = len(list(images_dir.glob("*")))
    label_count = len(list(labels_dir.glob("*.txt")))

    print(
        f"{split}: "
        f"{image_count} imagens | "
        f"{label_count} labels"
    )

train: 950 imagens | 950 labels
val: 251 imagens | 251 labels
test: 118 imagens | 118 labels


In [14]:
sample_labels = sorted(
    (SEGMENTATION_DIR / "labels" / "train").glob("*.txt")
)

print("Quantidade de labels:", len(sample_labels))

if sample_labels:
    sample_file = sample_labels[0]

    print("\nArquivo de exemplo:")
    print(sample_file)

    print("\nConteúdo:")
    print(sample_file.read_text(encoding="utf-8")[:1000])

Quantidade de labels: 950

Arquivo de exemplo:
c:\Users\Andrezinho\OneDrive\Documentos\GitHub\AeroVision\data\processed\drone_traffic_yolo\labels\train\seq3-drone_0000002_jpg.rf.0d2d3558435d8ec2e4f2069e4819247d.txt

Conteúdo:
0 0.186458 0.300926 0.185417 0.296296 0.184896 0.290741 0.184896 0.287037 0.183854 0.286111 0.182812 0.290741 0.181250 0.289815 0.180208 0.286111 0.179167 0.279630 0.177604 0.276852 0.176042 0.273148 0.173438 0.274074 0.172917 0.277778 0.172917 0.282407 0.174479 0.287963 0.175521 0.292593 0.176563 0.296296 0.176563 0.300926 0.177083 0.306481 0.178125 0.311111 0.179167 0.312037 0.179167 0.307407 0.179167 0.304630 0.180729 0.304630 0.181771 0.303704 0.183333 0.306481 0.184375 0.310185 0.185937 0.311111 0.187500 0.312037 0.188542 0.308333 0.187500 0.303704 0.186458 0.300926
2 0.062941 1.000000 0.051042 0.989815 0.039583 0.989815 0.030329 1.000000 0.062941 1.000000
2 0.269271 0.638889 0.273958 0.637037 0.277604 0.637963 0.281771 0.633333 0.283854 0.623148 0.302604 0.5

In [15]:
print("Configuração da segmentação:")
print(SEGMENTATION_CONFIG.read_text(encoding="utf-8"))

Configuração da segmentação:
path: ../data/processed/drone_traffic_yolo

train: images/train
val: images/val
test: images/test

names:
  0: bicycle
  1: bus
  2: car
  3: lorry


In [17]:
print("Configuração existe:", SEGMENTATION_CONFIG.exists())

Configuração existe: True


In [18]:
diagnostic_results = model.train(
    data=str(SEGMENTATION_CONFIG),
    epochs=1,
    imgsz=416,
    batch=2,
    workers=2,
    cache=False,
    device=0,
    project=str(PROJECT_DIR / "results" / "segmentation"),
    name="gpu_diagnostic",
)

Ultralytics 8.4.155  Python-3.11.0 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce GTX 1050 Ti, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\Andrezinho\OneDrive\Documentos\GitHub\AeroVision\configs\aerovision_segmentation.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937

In [19]:
batch_test_results = model.train(
    data=str(SEGMENTATION_CONFIG),
    epochs=1,
    imgsz=416,
    batch=4,
    workers=2,
    cache=False,
    device=0,
    project=str(PROJECT_DIR / "results" / "segmentation"),
    name="batch4_test",
)

print("Teste com batch=4 concluído.")

Ultralytics 8.4.155  Python-3.11.0 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce GTX 1050 Ti, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\Andrezinho\OneDrive\Documentos\GitHub\AeroVision\configs\aerovision_segmentation.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\Andrezinho\OneDrive\D

In [20]:
baseline_results = model.train(
    data=str(SEGMENTATION_CONFIG),
    epochs=30,
    imgsz=416,
    batch=4,
    workers=2,
    cache=False,
    device=0,
    project=str(PROJECT_DIR / "results" / "segmentation"),
    name="baseline",
)

Ultralytics 8.4.155  Python-3.11.0 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce GTX 1050 Ti, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\Andrezinho\OneDrive\Documentos\GitHub\AeroVision\configs\aerovision_segmentation.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\Andrezinho\OneDrive\

In [21]:
from pathlib import Path
from ultralytics import YOLO

BEST_SEGMENTATION_MODEL = (
    PROJECT_DIR
    / "results"
    / "segmentation"
    / "baseline"
    / "weights"
    / "best.pt"
)

print("Modelo:", BEST_SEGMENTATION_MODEL)
print("Existe:", BEST_SEGMENTATION_MODEL.exists())

best_segmentation_model = YOLO(
    str(BEST_SEGMENTATION_MODEL)
)

print("Modelo de segmentação carregado com sucesso.")

Modelo: c:\Users\Andrezinho\OneDrive\Documentos\GitHub\AeroVision\results\segmentation\baseline\weights\best.pt
Existe: True
Modelo de segmentação carregado com sucesso.


In [22]:
test_results = best_segmentation_model.val(
    data=str(SEGMENTATION_CONFIG),
    split="test",
    imgsz=416,
    batch=4,
    device=0,
    project=str(PROJECT_DIR / "results" / "segmentation"),
    name="evaluation"
)

print("Avaliação de segmentação no conjunto de teste concluída.")

Ultralytics 8.4.155  Python-3.11.0 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce GTX 1050 Ti, 4096MiB)
YOLO11n-seg summary (fused): 113 layers, 2,835,348 parameters, 0 gradients, 9.6 GFLOPs
WARNING val: Slow image access detected (ping: 0.50.3 ms, read: 18.66.8 MB/s, size: 26.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning C:\Users\Andrezinho\OneDrive\Documentos\GitHub\AeroVision\data\processed\drone_traffic_yolo\labels\test... 118 images, 15 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 118/118 229.3it/s 0.5s1s
val: New cache created: C:\Users\Andrezinho\OneDrive\Documentos\GitHub\AeroVision\data\processed\drone_traffic_yolo\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 30/30 12.2it/s 2.5s0.1s
                   all        118       1697      0.909      0.933     

In [23]:
EVALUATION_DIR = (
    PROJECT_DIR
    / "results"
    / "segmentation"
    / "evaluation"
)

print("Arquivos gerados:")
for file in sorted(EVALUATION_DIR.iterdir()):
    print(file.name)

Arquivos gerados:
BoxF1_curve.png
BoxP_curve.png
BoxPR_curve.png
BoxR_curve.png
confusion_matrix.png
confusion_matrix_normalized.png
MaskF1_curve.png
MaskP_curve.png
MaskPR_curve.png
MaskR_curve.png
val_batch0_labels.jpg
val_batch0_pred.jpg
val_batch1_labels.jpg
val_batch1_pred.jpg
val_batch2_labels.jpg
val_batch2_pred.jpg
